# Tutorial de datos con Python: pandas + NumPy + MongoDB

Un tutorial **autocontenido** para cualquier proyecto de análisis y carga de datos: leer archivos, limpiar, transformar, agrupar, unir, medir similitud, generar datos sintéticos y cargarlos a MongoDB con índices. Cubre todo lo necesario para un proyecto tipo *"integrar varias fuentes → vocabulario común → recomendador"*, pero sirve para cualquier otro.

**Cómo usarlo:** ejecuten las celdas en orden. Cada sección tiene (1) la idea, (2) código ejecutable con datos de juguete, (3) un recuadro de *errores comunes* y (4) ejercicios al final del notebook.

### Índice
- **Parte A. Fundamentos**: 1 Estructuras · 2 Leer/escribir · 3 Inspeccionar · 4 Seleccionar · 5 Filtrar
- **Parte B. Limpieza**: 6 Nulos · 7 Tipos · 8 Duplicados · 9 Texto · 10 Fechas · 11 Aplicar funciones
- **Parte C. Transformar**: 12 Listas y `explode` · 13 Mapeo/diccionarios · 14 Agrupar · 15 Unir · 16 Reestructurar · 17 Ordenar/ranking · 18 Ventanas
- **Parte D. Matemática y datos**: 19 NumPy · 20 Aleatorios y sintéticos · 21 Frecuencias, IDF, similitud
- **Parte E. Calidad y rendimiento**: 22 Validación · 23 Rendimiento
- **Parte F. MongoDB**: 24 Conceptos · 25 Preparar documentos · 26 pymongo CRUD · 27 Consultas y agregación · 28 Índices y `explain` · 29 Réplicas y sharding
- **Parte G. Cierre**: 30 Plantilla de pipeline · 31 Ejercicios · 32 Glosario de errores

In [1]:
import json, re, ast, math, unicodedata
from pathlib import Path
from io import StringIO
from collections import Counter

import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 160)

pandas 3.0.6 | numpy 2.5.3


> **Nota pandas 3.x:** el texto ahora se guarda con dtype `str` (antes `object`), y *Copy-on-Write* es el modo por defecto: modificar una vista **no** altera el original. Por eso no existe ya el viejo `SettingWithCopyWarning`; si quieres modificar, asigna el resultado: `df = df.algo(...)`. Evita `inplace=True` (no aporta nada y se está retirando).

# Parte A. Fundamentos

## 1. Estructuras: `Series` y `DataFrame`
- **Series**: columna con índice (1 dimensión).
- **DataFrame**: tabla de Series que comparten índice (2 dimensiones).
- **Index**: etiquetas de las filas. Por defecto 0,1,2… pero puede ser cualquier cosa.

In [2]:
# Series
s = pd.Series([10, 20, 30], index=["a", "b", "c"], name="valor")
print(s)
print(s["b"], s.iloc[0], s.values, s.index.tolist())

# DataFrame desde dict de columnas
df = pd.DataFrame({
    "id":      [1, 2, 3, 4],
    "nombre":  ["Ana", "Luis", "Eva", "Raúl"],
    "puntaje": [7.5, 8.0, np.nan, 6.1],
})
# DataFrame desde lista de dicts (así llegan los JSON / documentos de Mongo)
df2 = pd.DataFrame([{"id": 1, "x": "a"}, {"id": 2, "x": "b", "extra": True}])  # columnas faltantes -> NaN
# DataFrame desde lista de listas
df3 = pd.DataFrame([[1, "a"], [2, "b"]], columns=["id", "x"])
display(df); display(df2); display(df3)

a    10
b    20
c    30
Name: valor, dtype: int64
20 10 [10 20 30] ['a', 'b', 'c']


,id,nombre,puntaje
0,1,Ana,7.5
1,2,Luis,8.0
2,3,Eva,NaN
3,4,Raúl,6.1


,id,x,extra
0,1,a,NaN
1,2,b,True


,id,x
0,1,a
1,2,b


In [3]:
# Atributos esenciales
print(df.shape, df.ndim, df.size)   # (filas, columnas), dimensiones, nº de celdas
print(df.columns.tolist())
print(df.index)
print(df.dtypes)
print(len(df))                      # nº de filas
df.T                                 # transpuesta
df.to_numpy()                        # a matriz NumPy

(4, 3) 2 12
['id', 'nombre', 'puntaje']
RangeIndex(start=0, stop=4, step=1)
id           int64
nombre         str
puntaje    float64
dtype: object
4


array([[1, 'Ana', 7.5],
       [2, 'Luis', 8.0],
       [3, 'Eva', nan],
       [4, 'Raúl', 6.1]], dtype=object)

## 2. Leer y escribir archivos

| Formato | Leer | Escribir | Cuándo |
|---|---|---|---|
| CSV / TSV | `pd.read_csv` | `df.to_csv` | lo más común |
| CSV comprimido (`.zip`, `.gz`) | `pd.read_csv` (se infiere) | `to_csv(compression=...)` | datasets grandes |
| JSON (array) | `pd.read_json` | `to_json(orient="records")` | APIs |
| JSON Lines (1 objeto por línea) | `pd.read_json(lines=True)` | `to_json(orient="records", lines=True)` | datasets grandes, Mongo |
| Excel | `pd.read_excel` | `to_excel` | hojas de cálculo |
| Parquet | `pd.read_parquet` | `to_parquet` | rápido, conserva tipos |
| Pickle | `pd.read_pickle` | `to_pickle` | solo Python, temporal |

In [4]:
# --- Parámetros más útiles de read_csv (demo con texto en memoria) ---
texto = '''id;nombre;fecha;monto;extra
1;Ana;2026-01-05;10,5;N/A
2;Luis;2026-02-10;20,0;-
3;Eva;2026-03-15;;ok
'''
df_csv = pd.read_csv(
    StringIO(texto),
    sep=";",                    # separador
    decimal=",",                # coma decimal
    na_values=["N/A", "-"],     # textos que significan nulo
    parse_dates=["fecha"],      # a datetime al leer
    dtype={"id": "int64"},      # forzar tipos
    usecols=["id", "nombre", "fecha", "monto", "extra"],  # solo estas columnas
    # nrows=100,                # leer solo N filas (explorar rápido)
    # skiprows=1,               # saltar filas iniciales
    # header=None, names=[...], # si no hay encabezado
    # encoding="latin-1",       # si hay caracteres raros (por defecto utf-8)
    # on_bad_lines="skip",      # saltar filas rotas
)
display(df_csv); print(df_csv.dtypes)

,id,nombre,fecha,monto,extra
0,1,Ana,2026-01-05,10.5,NaN
1,2,Luis,2026-02-10,20.0,NaN
2,3,Eva,2026-03-15,NaN,ok


id                 int64
nombre               str
fecha     datetime64[us]
monto            float64
extra                str
dtype: object


In [5]:
# --- Escribir y leer de vuelta (en una carpeta temporal) ---
import tempfile
tmp = Path(tempfile.mkdtemp())

df.to_csv(tmp / "d.csv", index=False)                      # index=False: no guardar el índice
df.to_json(tmp / "d.jsonl", orient="records", lines=True, force_ascii=False)
df.to_json(tmp / "d.json", orient="records", force_ascii=False, indent=1)
df.to_csv(tmp / "d.csv.gz", index=False)                   # comprime por extensión
try:
    df.to_parquet(tmp / "d.parquet")                       # requiere pyarrow
except ImportError:
    print("pyarrow no instalado: pip install pyarrow")

print(sorted(p.name for p in tmp.iterdir()))
pd.read_json(tmp / "d.jsonl", lines=True)

['d.csv', 'd.csv.gz', 'd.json', 'd.jsonl', 'd.parquet']


,id,nombre,puntaje
0,1,Ana,7.5
1,2,Luis,8.0
2,3,Eva,NaN
3,4,Raúl,6.1


In [6]:
# --- Archivos grandes: leer por pedazos (chunksize) y acumular lo mínimo ---
grande = StringIO("g\n" + "\n".join(["a", "b", "a", "c"] * 1000))
conteo = Counter()
for trozo in pd.read_csv(grande, chunksize=1000):      # cada trozo es un DataFrame
    conteo.update(trozo["g"].value_counts().to_dict())
print(conteo)

# --- Rutas portables con pathlib ---
RAIZ = Path.cwd()
RAW = RAIZ / "datos" / "raw"
print(RAW, RAW.exists())
# RAW.mkdir(parents=True, exist_ok=True)   # crear carpeta
# list(RAW.glob("*.csv"))                  # listar archivos

Counter({'a': 2000, 'b': 1000, 'c': 1000})
/Users/andres/Desktop/gamecon/datos/raw True


**Errores comunes**
- `UnicodeDecodeError` → prueba `encoding="latin-1"` o `"utf-8-sig"`.
- Todo sale en una sola columna → el separador no es coma: usa `sep=";"` o `sep="\t"`.
- Un número llega como texto → hay valores raros (`"N/A"`): usa `na_values` o luego `pd.to_numeric(errors="coerce")`.
- Memoria insuficiente → `usecols`, `dtype`, `nrows`, `chunksize`.
- `.json.gz` con un objeto por línea → `lines=True`.

## 3. Inspeccionar (perfilar) un dataset
Lo primero que se hace con cualquier fuente nueva.

In [7]:
toy = pd.DataFrame({
    "nombre":   ["Alfa", "Beta", "Gamma", "Gamma", "Delta", "Epsilon", None],
    "empresa":  [" Acme ", "acme", "Zeta Co", "Zeta Co", "Omega", None, "Omega"],
    "anio":     [2001, 2005, "2010", "2010", np.nan, 1999, 2020],
    "generos":  ["Action, Fantasy", "Action", "Sci-Fi, Drama, Action", "Sci-Fi, Drama, Action", None, "Fantasy", "Drama"],
    "ventas":   [1.5, 0.3, 9.9, 9.9, 0.1, 2.2, 0.7],
})
toy

,nombre,empresa,anio,generos,ventas
0,Alfa,Acme,2001,"Action, Fantasy",1.5
1,Beta,acme,2005,Action,0.3
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
4,Delta,Omega,NaN,NaN,0.1
5,Epsilon,NaN,1999,Fantasy,2.2
6,NaN,Omega,2020,Drama,0.7


In [8]:
print(toy.shape)
toy.info()                                 # tipos, no-nulos, memoria
display(toy.head(3), toy.tail(2), toy.sample(2, random_state=0))
display(toy.describe())                    # numéricas
display(toy.describe(include="all"))       # todas
print(toy.isna().sum())                    # nulos por columna
print(toy.isna().mean().round(2))          # % de nulos
print("duplicadas exactas:", toy.duplicated().sum())
print(toy.nunique())                       # distintos por columna
print(toy["empresa"].value_counts(dropna=False))   # frecuencias (incluye nulos)
print(toy["empresa"].value_counts(normalize=True)) # proporciones
print(toy["ventas"].agg(["min", "max", "mean", "median", "std"]))
print(toy["ventas"].quantile([.25, .5, .75]))
print(toy.select_dtypes(include="number").columns.tolist())   # solo columnas numéricas
toy["generos"].unique()[:5]

(7, 5)
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   nombre   6 non-null      str    
 1   empresa  6 non-null      str    
 2   anio     6 non-null      object 
 3   generos  6 non-null      str    
 4   ventas   7 non-null      float64
dtypes: float64(1), object(1), str(3)
memory usage: 554.0+ bytes


,nombre,empresa,anio,generos,ventas
0,Alfa,Acme,2001,"Action, Fantasy",1.5
1,Beta,acme,2005,Action,0.3
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9


,nombre,empresa,anio,generos,ventas
5,Epsilon,NaN,1999,Fantasy,2.2
6,NaN,Omega,2020,Drama,0.7


,nombre,empresa,anio,generos,ventas
6,NaN,Omega,2020,Drama,0.7
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9


,ventas
count,7.000000
mean,3.514286
std,4.420569
min,0.100000
25%,0.500000
50%,1.500000
75%,6.050000
max,9.900000


,nombre,empresa,anio,generos,ventas
count,6,6,6,6,7.000000
unique,5,4,5,5,NaN
top,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",NaN
freq,2,2,2,2,NaN
mean,NaN,NaN,NaN,NaN,3.514286
std,NaN,NaN,NaN,NaN,4.420569
min,NaN,NaN,NaN,NaN,0.100000
25%,NaN,NaN,NaN,NaN,0.500000
50%,NaN,NaN,NaN,NaN,1.500000
75%,NaN,NaN,NaN,NaN,6.050000


nombre     1
empresa    1
anio       1
generos    1
ventas     0
dtype: int64
nombre     0.14
empresa    0.14
anio       0.14
generos    0.14
ventas     0.00
dtype: float64
duplicadas exactas: 1
nombre     5
empresa    4
anio       5
generos    5
ventas     6
dtype: int64
empresa
Zeta Co    2
Omega      2
 Acme      1
acme       1
NaN        1
Name: count, dtype: int64
empresa
Zeta Co    0.333333
Omega      0.333333
 Acme      0.166667
acme       0.166667
Name: proportion, dtype: float64
min       0.100000
max       9.900000
mean      3.514286
median    1.500000
std       4.420569
Name: ventas, dtype: float64
0.25    0.50
0.50    1.50
0.75    6.05
Name: ventas, dtype: float64
['ventas']


<ArrowStringArray>
['Action, Fantasy', 'Action', 'Sci-Fi, Drama, Action', nan, 'Fantasy']
Length: 5, dtype: str

In [9]:
# Función reutilizable de perfilado para CUALQUIER DataFrame
def perfil(d: pd.DataFrame, nombre: str = "dataset") -> pd.DataFrame:
    print(f"== {nombre}: {d.shape[0]} filas x {d.shape[1]} columnas | duplicadas: {d.duplicated().sum()}")
    return pd.DataFrame({
        "dtype":    d.dtypes.astype(str),
        "nulos":    d.isna().sum(),
        "%nulos":   (d.isna().mean() * 100).round(1),
        "distintos": d.nunique(),
        "ejemplo":  d.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
    })

perfil(toy, "toy")

== toy: 7 filas x 5 columnas | duplicadas: 1


,dtype,nulos,%nulos,distintos,ejemplo
nombre,str,1,14.3,5,Alfa
empresa,str,1,14.3,4,Acme
anio,object,1,14.3,5,2001
generos,str,1,14.3,5,"Action, Fantasy"
ventas,float64,0,0.0,6,1.5


## 4. Seleccionar columnas y filas
- `df["col"]` → Series · `df[["a","b"]]` → DataFrame
- `df.loc[filas, columnas]` → por **etiqueta** (el final se incluye)
- `df.iloc[filas, columnas]` → por **posición** (el final no se incluye)
- `df.at[fila, col]` / `df.iat[i, j]` → un solo valor (rápido)

In [10]:
toy["nombre"]
toy[["nombre", "empresa"]]
toy.loc[0:2, ["nombre", "ventas"]]
toy.iloc[0:2, 0:2]
toy.loc[toy["ventas"] > 1, "nombre"]       # filas por condición, columna concreta
toy.at[0, "nombre"]
toy.iloc[-1]                                # última fila
toy.drop(columns=["ventas"])                # quitar columnas (no modifica el original)
toy.filter(items=["nombre", "anio"])        # columnas por nombre
toy.filter(like="gen")                      # nombre contiene...
toy.filter(regex="^e")                      # nombre cumple regex
toy.reindex(columns=["ventas", "nombre"])   # reordenar/seleccionar
toy.set_index("nombre").loc["Beta"]         # usar una columna como índice

empresa      acme
anio         2005
generos    Action
ventas        0.3
Name: Beta, dtype: object

## 5. Filtrar con condiciones
Operadores: `&` (y), `|` (o), `~` (no). **Cada condición va entre paréntesis.** No uses `and`/`or`.

In [11]:
toy[toy["ventas"] > 1]
toy[(toy["ventas"] > 1) & (toy["empresa"].notna())]
toy[(toy["ventas"] < 0.5) | (toy["ventas"] > 5)]
toy[~toy["empresa"].isin(["Omega"])]
toy[toy["empresa"].isin(["Omega", "Zeta Co"])]
toy[toy["ventas"].between(0.5, 3)]
toy[toy["empresa"].notna()]                 # no nulos
toy[toy["generos"].str.contains("Action", na=False)]   # texto (na=False imprescindible)
toy.query("ventas > 1 and empresa == 'Omega'")         # sintaxis tipo texto
umbral = 2
toy.query("ventas > @umbral")               # @ usa variables de Python
toy.loc[lambda d: d["ventas"] > 5]          # con función (útil en cadenas)
toy.nlargest(2, "ventas")                   # top-N
toy.where(toy["ventas"] > 1)                # mantiene forma, pone NaN donde no cumple

,nombre,empresa,anio,generos,ventas
0,Alfa,Acme,2001,"Action, Fantasy",1.5
1,NaN,NaN,NaN,NaN,NaN
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
4,NaN,NaN,NaN,NaN,NaN
5,Epsilon,NaN,1999,Fantasy,2.2
6,NaN,NaN,NaN,NaN,NaN


**Errores comunes:** olvidar paréntesis (`a > 1 & b < 2` falla) · comparar con `NaN` usando `==` (usa `isna()`) · `str.contains` sin `na=False` cuando hay nulos.

# Parte B. Limpieza

## 6. Valores nulos
`NaN`/`None`/`pd.NA`/`NaT` son "faltantes". Decide para cada columna: **descartar fila**, **rellenar** o **dejar**.

In [12]:
n = pd.DataFrame({"a": [1, np.nan, 3, None], "b": ["x", None, "z", "w"], "c": [np.nan] * 4})
n.isna(); n.notna()
n.dropna()                         # quita filas con cualquier nulo
n.dropna(how="all")                # solo filas totalmente vacías
n.dropna(subset=["b"])             # solo mirando una columna
n.dropna(axis=1, how="all")        # quita columnas vacías
n.fillna({"a": 0, "b": "desconocido"})   # relleno distinto por columna
n["a"].fillna(n["a"].mean())       # con estadístico
n["a"].ffill()                     # propaga el valor anterior (bfill: el siguiente)
n["a"].interpolate()               # interpola numéricos
n.replace({"x": np.nan})           # convertir un valor en nulo

,a,b,c
0,1.0,NaN,NaN
1,NaN,NaN,NaN
2,3.0,z,NaN
3,NaN,w,NaN


## 7. Tipos de datos

In [13]:
t = pd.DataFrame({"n": ["1", "2", "x", None], "f": ["2026-01-01", "2026-02-01", "mal", None], "c": ["a", "b", "a", "a"]})
t["n_num"] = pd.to_numeric(t["n"], errors="coerce")        # lo inválido -> NaN
t["f_dt"]  = pd.to_datetime(t["f"], errors="coerce")       # lo inválido -> NaT
t["c_cat"] = t["c"].astype("category")                     # ahorra memoria si hay pocos valores distintos
t["n_int"] = t["n_num"].astype("Int64")                    # entero que admite nulos (I mayúscula)
t["s"]     = t["c"].astype("string")                       # texto
t["bool"]  = t["c"].eq("a")                                # booleano
print(t.dtypes)
t.convert_dtypes().dtypes                                  # elige los mejores tipos automáticamente

n                   str
f                   str
c                   str
n_num           float64
f_dt     datetime64[us]
c_cat          category
n_int             Int64
s                string
bool               bool
dtype: object


n                string
f                string
c                string
n_num             Int64
f_dt     datetime64[us]
c_cat          category
n_int             Int64
s                string
bool            boolean
dtype: object

## 8. Duplicados

In [14]:
toy.duplicated()                                    # True en la 2ª aparición
toy.duplicated(keep=False)                          # marca TODAS las repetidas
toy.duplicated(subset=["nombre"])                   # según columnas
toy.drop_duplicates()                               # filas idénticas
toy.drop_duplicates(subset=["nombre"], keep="last") # primero/último
# Ver qué se repite antes de borrar:
toy[toy.duplicated(subset=["nombre"], keep=False)].sort_values("nombre")

,nombre,empresa,anio,generos,ventas
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9


## 9. Texto con `.str`
Se aplica a toda una Series a la vez. Con nulos, devuelven nulo (no fallan).

In [15]:
t = pd.Series(["  Role-Playing ", "SCI-FI", "Slice of Life", None, "Sports/Racing", "Acción y Aventura"])
print(t.str.strip().tolist())                   # strip/lstrip/rstrip
print(t.str.lower().tolist())                   # upper, title, capitalize
print(t.str.len().tolist())
print(t.str.replace("-", " ", regex=False).tolist())          # literal
print(t.str.replace(r"\s+", " ", regex=True).tolist())        # regex
print(t.str.contains("sci", case=False, na=False).tolist())
print(t.str.startswith("Sl", na=False).tolist(), t.str.endswith("ng", na=False).tolist())
print(t.str.split("/").tolist())                # -> listas
print(t.str.split(" ", expand=True))            # -> columnas
print(t.str.extract(r"(\w+)-(\w+)"))           # grupos de regex -> columnas
print(t.str.findall(r"[A-Z]"))                  # todas las coincidencias
print(t.str[:4].tolist())                       # slicing
print(t.str.cat(sep=" | ", na_rep=""))          # unir todo
print(t.str.zfill(3).head(2).tolist(), t.str.pad(12, fillchar="*").head(2).tolist())

['Role-Playing', 'SCI-FI', 'Slice of Life', nan, 'Sports/Racing', 'Acción y Aventura']
['  role-playing ', 'sci-fi', 'slice of life', nan, 'sports/racing', 'acción y aventura']
[15.0, 6.0, 13.0, nan, 13.0, 17.0]
['  Role Playing ', 'SCI FI', 'Slice of Life', nan, 'Sports/Racing', 'Acción y Aventura']
[' Role-Playing ', 'SCI-FI', 'Slice of Life', nan, 'Sports/Racing', 'Acción y Aventura']
[False, True, False, False, False, False]
[False, False, True, False, False, False] [False, False, False, False, True, False]
[['  Role-Playing '], ['SCI-FI'], ['Slice of Life'], nan, ['Sports', 'Racing'], ['Acción y Aventura']]
               0    1             2    3
0                      Role-Playing     
1         SCI-FI  NaN           NaN  NaN
2          Slice   of          Life  NaN
3            NaN  NaN           NaN  NaN
4  Sports/Racing  NaN           NaN  NaN
5         Acción    y      Aventura  NaN
      0        1
0  Role  Playing
1   SCI       FI
2   NaN      NaN
3   NaN      NaN
4   NaN 

In [16]:
# Quitar tildes y normalizar: patrón completo y reutilizable
def quitar_tildes(x):
    if not isinstance(x, str):
        return x
    return "".join(c for c in unicodedata.normalize("NFKD", x) if not unicodedata.combining(c))

def normalizar(serie: pd.Series) -> pd.Series:
    """minúsculas, sin tildes, sin espacios sobrantes, separadores -> guion."""
    return (serie.map(quitar_tildes)
                 .str.strip()
                 .str.lower()
                 .str.replace(r"[\s_/]+", "-", regex=True)
                 .str.replace(r"[^a-z0-9\-]", "", regex=True)
                 .str.strip("-"))

normalizar(t)

0         role-playing
1               sci-fi
2        slice-of-life
3                  NaN
4        sports-racing
5    accion-y-aventura
dtype: str

**Errores comunes:** `.str` sobre una columna que no es texto (convierte con `.astype("string")`) · `replace` sin `regex=False` cuando el patrón tiene `.` o `(` · normalizar **después** de agrupar (hazlo antes: `"Acme"` y `"acme"` serían grupos distintos).

## 10. Fechas y horas (`datetime`, `.dt`)

In [17]:
f = pd.to_datetime(pd.Series(["2026-11-01 10:00", "2026-11-01 12:30", "2026-11-02 09:15"]))
print(f.dt.date.tolist(), f.dt.hour.tolist(), f.dt.day_name().tolist())  # year, month, day, minute, dayofweek...
print((f + pd.Timedelta(hours=1, minutes=30)).tolist())                  # sumar duración
print((f.iloc[1] - f.iloc[0]))                                           # resta -> Timedelta
print(f.dt.floor("h").tolist())                                          # redondear (h = hora)
print(pd.date_range("2026-11-01 10:00", periods=4, freq="2h").tolist())  # secuencia
print(pd.to_datetime("01/11/2026", format="%d/%m/%Y"))                   # formato explícito
print(pd.Timestamp("2026-11-01 10:00") + pd.to_timedelta(np.array([30, 90]), unit="m"))
# ¿ocurre entre dos instantes? (solapamiento de eventos)
ini, fin = pd.Timestamp("2026-11-01 11:00"), pd.Timestamp("2026-11-01 13:00")
f[(f >= ini) & (f <= fin)]

[datetime.date(2026, 11, 1), datetime.date(2026, 11, 1), datetime.date(2026, 11, 2)] [10, 12, 9] ['Sunday', 'Sunday', 'Monday']
[Timestamp('2026-11-01 11:30:00'), Timestamp('2026-11-01 14:00:00'), Timestamp('2026-11-02 10:45:00')]
0 days 02:30:00
[Timestamp('2026-11-01 10:00:00'), Timestamp('2026-11-01 12:00:00'), Timestamp('2026-11-02 09:00:00')]
[Timestamp('2026-11-01 10:00:00'), Timestamp('2026-11-01 12:00:00'), Timestamp('2026-11-01 14:00:00'), Timestamp('2026-11-01 16:00:00')]
2026-11-01 00:00:00
DatetimeIndex(['2026-11-01 10:30:00', '2026-11-01 11:30:00'], dtype='datetime64[us]', freq=None)


1   2026-11-01 12:30:00
dtype: datetime64[us]

## 11. Aplicar funciones: `map`, `apply`, `agg`, `pipe`, `assign`
| Herramienta | Entra | Úsala para |
|---|---|---|
| `Series.map(dict_o_func)` | cada elemento | mapear valores (rápido con dict) |
| `Series.apply(func)` | cada elemento | lógica propia por elemento |
| `DataFrame.apply(func, axis=1)` | cada **fila** | lógica que mezcla columnas (lento) |
| `DataFrame.apply(func, axis=0)` | cada columna | resumen por columna |
| `.pipe(func)` | el DataFrame completo | encadenar funciones |
| `.assign(col=...)` | — | crear columnas sin modificar el original |
| `np.where`, `np.select` | vectorizado | condiciones (lo más rápido) |

In [18]:
d = pd.DataFrame({"a": [1, 2, 3, 4], "b": [10, 20, 30, 40], "g": ["x", "y", "x", "z"]})
print(d["g"].map({"x": "X", "y": "Y"}).tolist())            # no encontrado -> NaN
print(d["g"].map({"x": "X"}).fillna(d["g"]).tolist())       # dejar el original si no está
print(d["a"].apply(lambda v: v ** 2).tolist())
print(d.apply(lambda fila: fila["a"] + fila["b"], axis=1).tolist())
print(d[["a", "b"]].apply(np.sum))                           # por columna
d2 = (d.assign(suma=lambda x: x["a"] + x["b"])              # encadenar sin ensuciar
        .pipe(lambda x: x[x["suma"] > 15])
        .rename(columns={"g": "grupo"}))
display(d2)

# Condiciones vectorizadas (preferir sobre apply)
d["tam"] = np.where(d["a"] > 2, "grande", "chico")
d["nivel"] = np.select([d["a"] < 2, d["a"] < 4], ["bajo", "medio"], default="alto")
d["a"].clip(lower=2, upper=3)
pd.cut(d["b"], bins=[0, 15, 35, 100], labels=["bajo", "medio", "alto"])   # por rangos fijos
pd.qcut(d["b"], q=2, labels=["mitad baja", "mitad alta"])                 # por cuantiles
d

['X', 'Y', 'X', nan]
['X', 'y', 'X', 'z']
[1, 4, 9, 16]
[11, 22, 33, 44]
a     10
b    100
dtype: int64


,a,b,grupo,suma
1,2,20,y,22
2,3,30,x,33
3,4,40,z,44


,a,b,g,tam,nivel
0,1,10,x,chico,bajo
1,2,20,y,chico,medio
2,3,30,x,grande,medio
3,4,40,z,grande,alto


# Parte C. Transformar

## 12. Columnas con varios valores: listas, `split`, `explode`, dummies
Muy común: una celda contiene `"Action, Fantasy"` o `['a','b']`. Para contar/agrupar hay que separar.

Ruta típica: **texto → lista → limpiar elementos → `explode` → trabajar → reagrupar con `agg(list)`**.

In [19]:
g = toy[["nombre", "generos"]].dropna().drop_duplicates().copy()
g["lista"] = g["generos"].str.split(",")                              # texto -> lista
g["lista"] = g["lista"].map(lambda xs: [x.strip().lower() for x in xs])  # limpiar elementos
largo = g.explode("lista").rename(columns={"lista": "genero"})        # 1 fila por elemento
display(largo)

# Reagrupar a listas SIN repetidos y ordenadas
largo.groupby("nombre")["genero"].agg(lambda s: sorted(set(s))).reset_index()

,nombre,generos,genero
0,Alfa,"Action, Fantasy",action
0,Alfa,"Action, Fantasy",fantasy
1,Beta,Action,action
2,Gamma,"Sci-Fi, Drama, Action",sci-fi
2,Gamma,"Sci-Fi, Drama, Action",drama
2,Gamma,"Sci-Fi, Drama, Action",action
5,Epsilon,Fantasy,fantasy


,nombre,genero
0,Alfa,"[action, fantasy]"
1,Beta,[action]
2,Epsilon,[fantasy]
3,Gamma,"[action, drama, sci-fi]"


In [20]:
# Dummies / one-hot: una columna 0/1 por valor
g["generos"].str.get_dummies(sep=", ")
pd.get_dummies(largo["genero"], dtype=int)

# Si el archivo trae listas como TEXTO "['a','b']":
pd.Series(["['a', 'b']", "['c']"]).map(ast.literal_eval)

# Unir columnas de listas (ej: géneros + temas + demografías) en una sola lista:
x = pd.DataFrame({"gen": [["a", "b"], ["c"]], "tem": [["d"], []], "dem": [["e"], ["f"]]})
x["todo"] = x["gen"] + x["tem"] + x["dem"]          # + concatena listas fila a fila
x["todo"] = x["todo"].map(lambda xs: sorted(set(xs)))
display(x)

# Datos anidados (JSON): dict -> columnas
anid = pd.DataFrame({"id": [1, 2], "info": [{"x": 1, "y": {"z": 9}}, {"x": 3, "y": {"z": 7}}]})
pd.json_normalize(anid["info"])                      # y.z queda como columna "y.z"
pd.json_normalize([{"id": 1, "libros": [{"t": "A"}, {"t": "B"}]}], record_path="libros", meta=["id"])
# Contar elementos de listas
x["todo"].map(len)
# Aplanar y contar todos los elementos de una columna de listas
pd.Series([e for lst in x["todo"] for e in lst]).value_counts()

,gen,tem,dem,todo
0,"[a, b]",[d],[e],"[a, b, d, e]"
1,[c],[],[f],"[c, f]"


a    1
b    1
d    1
e    1
c    1
f    1
Name: count, dtype: int64

**Errores comunes:** `explode` deja índices repetidos (usa `reset_index(drop=True)` si molesta) · una celda `[]` vacía se convierte en `NaN` tras `explode` (filtra con `dropna`) · `list + NaN` falla: rellena nulos con `[]` antes (`df["c"].map(lambda v: v if isinstance(v, list) else [])`).

## 13. Mapeo con diccionarios y tablas de equivalencia
Para unificar vocabularios distintos en uno común: una tabla `origen → destino`.

In [21]:
mapeo = pd.DataFrame({
    "origen": ["sci-fi", "science fiction", "role-playing", "fantasy", "action"],
    "tag":    ["ciencia-ficcion", "ciencia-ficcion", "rol", "rol", "accion"],
    "nivel":  ["general", "general", "general", "especifico", "general"],
})
dic = dict(zip(mapeo["origen"], mapeo["tag"]))        # tabla -> dict

largo2 = largo.copy()
largo2["tag"] = largo2["genero"].map(dic)             # A) map: 1 destino por origen
display(largo2)

# B) merge: permite traer varias columnas y MÚLTIPLES destinos por origen (duplica filas a propósito)
con_tag = largo2.drop(columns="tag").merge(mapeo, left_on="genero", right_on="origen", how="left")
display(con_tag)

# Auditoría: ¿qué quedó sin mapear?  (esto guía cómo completar la tabla)
print(con_tag.loc[con_tag["tag"].isna(), "genero"].value_counts())

# Tabla de mapeo guardada como CSV y recargada:
mapeo.to_csv(tmp / "mapeo.csv", index=False)
pd.read_csv(tmp / "mapeo.csv").set_index("origen")["tag"].to_dict()

# Inverso: destino -> lista de orígenes
mapeo.groupby("tag")["origen"].agg(list)

,nombre,generos,genero,tag
0,Alfa,"Action, Fantasy",action,accion
0,Alfa,"Action, Fantasy",fantasy,rol
1,Beta,Action,action,accion
2,Gamma,"Sci-Fi, Drama, Action",sci-fi,ciencia-ficcion
2,Gamma,"Sci-Fi, Drama, Action",drama,NaN
2,Gamma,"Sci-Fi, Drama, Action",action,accion
5,Epsilon,Fantasy,fantasy,rol


,nombre,generos,genero,origen,tag,nivel
0,Alfa,"Action, Fantasy",action,action,accion,general
1,Alfa,"Action, Fantasy",fantasy,fantasy,rol,especifico
2,Beta,Action,action,action,accion,general
3,Gamma,"Sci-Fi, Drama, Action",sci-fi,sci-fi,ciencia-ficcion,general
4,Gamma,"Sci-Fi, Drama, Action",drama,NaN,NaN,NaN
5,Gamma,"Sci-Fi, Drama, Action",action,action,accion,general
6,Epsilon,Fantasy,fantasy,fantasy,rol,especifico


genero
drama    1
Name: count, dtype: int64


tag
accion                              [action]
ciencia-ficcion    [sci-fi, science fiction]
rol                  [role-playing, fantasy]
Name: origen, dtype: object

## 14. Agrupar y agregar (`groupby`)
Patrón **dividir → aplicar → combinar**. Sirve para "una fila por X" (por ejemplo, un registro por productor).

In [22]:
v = pd.DataFrame({
    "empresa": ["acme", "acme", "zeta", "zeta", "zeta", "omega"],
    "tag":     ["accion", "rol", "accion", "terror", "accion", "rol"],
    "ventas":  [1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
})
v.groupby("empresa").size()                              # filas por grupo
v.groupby("empresa")["ventas"].sum()
v.groupby("empresa")["ventas"].agg(["sum", "mean", "count"])
v.groupby("empresa")["tag"].nunique()                    # distintos por grupo

# Agregación NOMBRADA: la forma más clara de construir una tabla final
res = (v.groupby("empresa")
         .agg(n_filas=("tag", "size"),
              total=("ventas", "sum"),
              tags=("tag", lambda s: sorted(set(s))))    # lista sin repetidos
         .reset_index())
display(res)

v.groupby(["empresa", "tag"]).size().unstack(fill_value=0)       # tabla cruzada
pd.crosstab(v["empresa"], v["tag"])                              # equivalente directo
v.pivot_table(index="empresa", columns="tag", values="ventas", aggfunc="sum", fill_value=0)
v["total_grupo"] = v.groupby("empresa")["ventas"].transform("sum")   # valor del grupo en cada fila
v["frec_tag"]    = v["tag"].map(v["tag"].value_counts())            # frecuencia global pegada a cada fila
v["rank_en_grupo"] = v.groupby("empresa")["ventas"].rank(ascending=False)
v.groupby("empresa").head(1)                 # primera fila de cada grupo (tail, nth también)
v.groupby("empresa").filter(lambda g: len(g) >= 2)   # solo grupos con >=2 filas
v

,empresa,n_filas,total,tags
0,acme,2,3.0,"[accion, rol]"
1,omega,1,6.0,[rol]
2,zeta,3,12.0,"[accion, terror]"


,empresa,tag,ventas,total_grupo,frec_tag,rank_en_grupo
0,acme,accion,1.0,3.0,3,2.0
1,acme,rol,2.0,3.0,2,1.0
2,zeta,accion,3.0,12.0,3,3.0
3,zeta,terror,4.0,12.0,1,2.0
4,zeta,accion,5.0,12.0,3,1.0
5,omega,rol,6.0,6.0,2,1.0


**Errores comunes:** olvidar `reset_index()` (el grupo queda como índice) · `groupby` descarta filas con clave nula (usa `dropna=False` si las quieres) · agrupar **antes** de normalizar la clave.

## 15. Unir tablas: `merge`, `join`, `concat`
- **`merge`** = SQL JOIN (une por columnas clave): `inner`, `left`, `right`, `outer`.
- **`concat`** = apilar tablas (`axis=0`) o pegarlas lado a lado (`axis=1`).

In [23]:
a = pd.DataFrame({"k": [1, 2, 3], "x": ["a", "b", "c"]})
b = pd.DataFrame({"k": [2, 3, 4], "y": ["B", "C", "D"]})

a.merge(b, on="k", how="inner")        # solo coincidentes
a.merge(b, on="k", how="left")         # todo lo de a
a.merge(b, on="k", how="outer", indicator=True)   # _merge: left_only / right_only / both
a.merge(b, left_on="k", right_on="k", suffixes=("_a", "_b"))
a.merge(b, on="k", how="left", validate="one_to_one")  # ERROR si la clave no es única: protege de duplicados
a.set_index("k").join(b.set_index("k"), how="left")

# Antes/después: ¿el merge multiplicó filas?
n0 = len(a); n1 = len(a.merge(b, on="k", how="left")); print(n0, n1)

# Filas de a que NO están en b (anti-join)
a[~a["k"].isin(b["k"])]

# concat: apilar fuentes con las mismas columnas
s1 = pd.DataFrame({"nombre": ["A"], "dominio": ["juego"]})
s2 = pd.DataFrame({"nombre": ["B"], "dominio": ["anime"]})
pd.concat([s1, s2], ignore_index=True)
pd.concat([s1, s2], keys=["f1", "f2"])                 # recuerda de qué tabla vino
pd.concat([s1, s2.rename(columns={"nombre": "n"})])    # columnas distintas -> NaN (cuidado)
pd.concat([a, b], axis=1)                              # lado a lado por índice

3 3


,k,x,k,y
0,1,a,2,B
1,2,b,3,C
2,3,c,4,D


**Errores comunes:** claves duplicadas multiplican filas · claves con distinto tipo (`"1"` vs `1`) no coinciden · columnas con distinto nombre al apilar generan columnas NaN: renómbralas antes con `rename`.

## 16. Reestructurar: `melt`, `pivot`, `stack`

In [24]:
ancho = pd.DataFrame({"id": [1, 2], "accion": [1, 0], "rol": [1, 1]})
largo_m = ancho.melt(id_vars="id", var_name="tag", value_name="tiene")   # ancho -> largo
display(largo_m)
largo_m.pivot(index="id", columns="tag", values="tiene")                  # largo -> ancho
largo_m.query("tiene == 1")                                               # quedarse con los 1
ancho.set_index("id").stack()                                            # similar a melt, con índice

,id,tag,tiene
0,1,accion,1
1,2,accion,0
2,1,rol,1
3,2,rol,1


id        
1   accion    1
    rol       1
2   accion    0
    rol       1
dtype: int64

## 17. Ordenar, ranking y top-N

In [25]:
r = pd.DataFrame({"n": ["a", "b", "c", "d"], "p": [3, 1, 3, 2], "q": [9, 8, 7, 6]})
r.sort_values("p", ascending=False)
r.sort_values(["p", "q"], ascending=[False, True])      # varios criterios
r.sort_values("n", key=lambda s: s.str.lower())         # orden con transformación
r.sort_index()
r["rank"] = r["p"].rank(ascending=False, method="min")  # min / dense / first / average
r.nlargest(2, "p"); r.nsmallest(2, "q")                 # top / bottom N
r.sort_values("p", ascending=False).head(2)
r.reset_index(drop=True)

,n,p,q,rank
0,a,3,9,1.0
1,b,1,8,4.0
2,c,3,7,1.0
3,d,2,6,3.0


## 18. Ventanas y acumulados

In [26]:
w = pd.DataFrame({"dia": pd.date_range("2026-11-01", periods=5), "v": [1, 3, 2, 5, 4]})
w["acum"]  = w["v"].cumsum()                       # cumprod, cummax, cummin
w["media3"] = w["v"].rolling(3).mean()             # media móvil
w["dif"]   = w["v"].diff()                         # diferencia con la fila anterior
w["ant"]   = w["v"].shift(1)                       # desplazar
w["pct"]   = w["v"].pct_change()
w.set_index("dia").resample("2D")["v"].sum()       # reagrupar por periodos
w

,dia,v,acum,media3,dif,ant,pct
0,2026-11-01,1,1,NaN,NaN,NaN,NaN
1,2026-11-02,3,4,NaN,2.0,1.0,2.000000
2,2026-11-03,2,6,2.000000,-1.0,3.0,-0.333333
3,2026-11-04,5,11,3.333333,3.0,2.0,1.500000
4,2026-11-05,4,15,3.666667,-1.0,5.0,-0.200000


# Parte D. Matemática, datos sintéticos y similitud

## 19. NumPy esencial

In [27]:
a = np.array([1, 2, 3, 4]); M = np.arange(6).reshape(2, 3)
print(a.mean(), a.std(), a.sum(), a.max(), a.argmax())
print(a * 2, a ** 2, a + a)                       # vectorizado: sin bucles
print(M.shape, M.T.shape, M.sum(axis=0), M.sum(axis=1))
print(np.log(a), np.log1p(a), np.sqrt(a), np.exp(1))
print(np.dot(a, a), np.linalg.norm(a))            # producto punto, norma
print(np.unique(["b", "a", "b"], return_counts=True))
print(np.isin([1, 5], a), np.where(a > 2, "alto", "bajo"))
print(np.zeros(3), np.ones(3), np.full(3, 7), np.linspace(0, 1, 5))
print(np.round(3.14159, 2), np.clip(a, 2, 3))

2.5 1.118033988749895 10 4 3
[2 4 6 8] [ 1  4  9 16] [2 4 6 8]
(2, 3) (3, 2) [3 5 7] [ 3 12]
[0.         0.69314718 1.09861229 1.38629436] [0.69314718 1.09861229 1.38629436 1.60943791] [1.         1.41421356 1.73205081 2.        ] 2.718281828459045
30 5.477225575051661
(array(['a', 'b'], dtype='<U1'), array([1, 2]))
[ True False] ['bajo' 'bajo' 'alto' 'alto']
[0. 0. 0.] [1. 1. 1.] [7 7 7] [0.   0.25 0.5  0.75 1.  ]
3.14 [2 2 3 3]


## 20. Aleatorios y datos sintéticos
Usa **siempre** un generador con semilla (`default_rng(seed)`) para que los resultados sean reproducibles.

In [28]:
rng = np.random.default_rng(42)
print(rng.integers(0, 10, size=5))                 # enteros [0,10)
print(rng.random(3).round(3))                      # flotantes [0,1)
print(rng.normal(loc=0, scale=1, size=3).round(3)) # normal
print(rng.choice(["a", "b", "c"], size=5))                       # con reemplazo
print(rng.choice(["a", "b", "c"], size=2, replace=False))        # sin reemplazo
print(rng.choice(["a", "b", "c"], size=8, p=[0.7, 0.2, 0.1]))    # con pesos
print(rng.zipf(2.0, size=5))                       # cola larga (pocos muy populares)
print(rng.permutation(5))                          # barajar

[0 7 6 4 4]
[0.697 0.094 0.976]
[ 0.128 -0.316 -0.017]
['c' 'c' 'b' 'b' 'b']
['a' 'c']
['b' 'a' 'a' 'a' 'a' 'b' 'a' 'b']
[ 1  1  1 30  1]
[4 3 2 0 1]


In [29]:
# --- Ejemplo completo: asistentes, catálogo e interacciones sintéticas ---
rng = np.random.default_rng(7)
vocab = ["accion", "aventura", "rol", "terror", "ciencia-ficcion", "misterio", "deportes", "estrategia"]

stands = pd.DataFrame({
    "_id": [f"stand_{i}" for i in range(6)],
    "tags": [list(rng.choice(vocab, size=rng.integers(1, 4), replace=False)) for _ in range(6)],
})

asistentes = pd.DataFrame({
    "_id": [f"asis_{i}" for i in range(5)],
    "intereses_declarados": [list(rng.choice(vocab, size=rng.integers(1, 9), replace=False)) for _ in range(5)],
})

# Popularidad desigual: unos pocos stands reciben casi todo el tráfico (ley de potencias)
pesos = 1 / np.arange(1, len(stands) + 1)          # 1, 1/2, 1/3...
pesos = pesos / pesos.sum()                         # normalizar a 1
N = 20
interacciones = pd.DataFrame({
    "asistente_id": rng.choice(asistentes["_id"], size=N),
    "objetivo_id":  rng.choice(stands["_id"], size=N, p=pesos),
    "tipo":         rng.choice(["visita", "asistencia", "favorito"], size=N, p=[0.6, 0.25, 0.15]),
    "fecha":        pd.Timestamp("2026-11-01 10:00") + pd.to_timedelta(rng.integers(0, 600, size=N), unit="m"),
})
display(asistentes.head(2))
print(interacciones["objetivo_id"].value_counts())   # ¿se ve la desigualdad?
interacciones.sort_values("fecha").head(3)

,_id,intereses_declarados
0,asis_0,"[misterio, terror, estrategia, deportes, ciencia-ficcion]"
1,asis_1,"[estrategia, deportes, terror, rol, misterio, ciencia-fi..."


objetivo_id
stand_0    7
stand_2    4
stand_5    3
stand_1    3
stand_4    2
stand_3    1
Name: count, dtype: int64


,asistente_id,objetivo_id,tipo,fecha
2,asis_2,stand_1,visita,2026-11-01 10:18:00
17,asis_1,stand_3,asistencia,2026-11-01 10:34:00
11,asis_3,stand_0,visita,2026-11-01 10:47:00


In [30]:
# Contador tag -> n a partir de eventos (perfil deducido):
favs = interacciones.query("tipo == 'favorito'").merge(
    stands[["_id", "tags"]].rename(columns={"_id": "objetivo_id"}), on="objetivo_id")
favs = favs.explode("tags")
deducidos = (favs.groupby(["asistente_id", "tags"]).size()
                 .reset_index(name="n")
                 .groupby("asistente_id")
                 .apply(lambda g: dict(zip(g["tags"], g["n"])), include_groups=False))
deducidos

# Otras utilidades de muestreo
stands.sample(2, random_state=1)           # filas al azar
stands.sample(frac=0.5, random_state=1)    # fracción
stands.sample(10, replace=True, random_state=1)   # bootstrap

,_id,tags
5,stand_5,"[rol, ciencia-ficcion, terror]"
3,stand_3,[terror]
4,stand_4,"[rol, aventura, estrategia]"
0,stand_0,"[terror, estrategia, ciencia-ficcion]"
1,stand_1,"[rol, accion, aventura]"
3,stand_3,[terror]
5,stand_5,"[rol, ciencia-ficcion, terror]"
0,stand_0,"[terror, estrategia, ciencia-ficcion]"
0,stand_0,"[terror, estrategia, ciencia-ficcion]"
1,stand_1,"[rol, accion, aventura]"


## 21. Frecuencias, IDF y medidas de similitud
Ideas clave para recomendar "lo más parecido":
- **Frecuencia (df)**: en cuántos ítems aparece cada etiqueta.
- **IDF**: pesa más las etiquetas raras. `idf = log(N / df)` (o suavizado `log((1+N)/(1+df)) + 1`).
- **Jaccard**: `|A ∩ B| / |A ∪ B|` sobre conjuntos (0 a 1).
- **Jaccard ponderado**: suma de pesos de la intersección / suma de pesos de la unión.
- **Coseno**: ángulo entre vectores (con pesos por etiqueta).

In [31]:
# 1) Frecuencia de cada etiqueta (en cuántos ítems aparece; usa set para no contar doble)
items = pd.DataFrame({"item": ["S1", "S2", "S3", "S4"],
                      "tags": [["accion", "rol"], ["accion"], ["terror", "accion"], ["rol", "misterio"]]})
N = len(items)
frec = (items.explode("tags").drop_duplicates(["item", "tags"])
             .groupby("tags").size().rename("df").reset_index())
frec["idf"] = np.log(N / frec["df"])
frec["idf_suave"] = np.log((1 + N) / (1 + frec["df"])) + 1
display(frec.sort_values("df", ascending=False))
idf = frec.set_index("tags")["idf_suave"].to_dict()

,tags,df,idf,idf_suave
0,accion,3,0.287682,1.223144
2,rol,2,0.693147,1.510826
1,misterio,1,1.386294,1.916291
3,terror,1,1.386294,1.916291


In [32]:
# 2) Jaccard simple y ponderado
def jaccard(a, b):
    a, b = set(a), set(b)
    u = a | b
    return len(a & b) / len(u) if u else 0.0

def jaccard_pond(a, b, w):
    a, b = set(a), set(b)
    num = sum(w.get(t, 1.0) for t in a & b)
    den = sum(w.get(t, 1.0) for t in a | b)
    return num / den if den else 0.0

def coseno(a, b, w):
    # vectores binarios ponderados por idf
    a, b = set(a), set(b)
    num = sum(w.get(t, 1.0) ** 2 for t in a & b)
    na = math.sqrt(sum(w.get(t, 1.0) ** 2 for t in a))
    nb = math.sqrt(sum(w.get(t, 1.0) ** 2 for t in b))
    return num / (na * nb) if na and nb else 0.0

gustos = ["accion", "misterio"]
items["jaccard"]  = items["tags"].map(lambda t: jaccard(gustos, t))
items["jacc_idf"] = items["tags"].map(lambda t: jaccard_pond(gustos, t, idf))
items["coseno"]   = items["tags"].map(lambda t: coseno(gustos, t, idf))
items.sort_values("jacc_idf", ascending=False).round(3)

,item,tags,jaccard,jacc_idf,coseno
3,S4,"[rol, misterio]",0.333,0.412,0.662
1,S2,[accion],0.500,0.390,0.538
0,S1,"[accion, rol]",0.333,0.263,0.339
2,S3,"[terror, accion]",0.333,0.242,0.289


In [33]:
# 3) Combinar fuentes de gusto: declarado pesa más que deducido
declarados = ["accion", "rol"]
deducido = {"accion": 3, "terror": 1}        # tag -> contador
perfil_w = {}
for t in declarados:   perfil_w[t] = perfil_w.get(t, 0) + 1.0
for t, c in deducido.items(): perfil_w[t] = perfil_w.get(t, 0) + 0.3 * c   # peso 0.3 por favorito
print(perfil_w)

def puntaje(tags_item, perfil_w, idf):
    # suma de pesos de perfil * idf para las etiquetas que el item comparte
    return sum(perfil_w.get(t, 0) * idf.get(t, 1.0) for t in set(tags_item))

items["puntaje"] = items["tags"].map(lambda t: puntaje(t, perfil_w, idf))
items.sort_values("puntaje", ascending=False)[["item", "tags", "puntaje"]]

{'accion': 1.9, 'rol': 1.0, 'terror': 0.3}


,item,tags,puntaje
0,S1,"[accion, rol]",3.834798
2,S3,"[terror, accion]",2.898860
1,S2,[accion],2.323973
3,S4,"[rol, misterio]",1.510826


In [34]:
# 4) Versión matricial (rápida con muchos ítems): vectores 0/1 y producto punto
M = items["tags"].explode().str.get_dummies().groupby(level=0).sum()   # ítems x tags (0/1)
cols = M.columns
w_vec = np.array([idf.get(c, 1.0) for c in cols])
u_vec = np.array([1.0 if c in gustos else 0.0 for c in cols])
num = (M.values * w_vec) @ u_vec                                        # @ = producto matricial
den = np.sqrt(((M.values * w_vec) ** 2).sum(axis=1)) * np.sqrt(((u_vec * w_vec) ** 2).sum())
M.assign(coseno_matricial=np.divide(num, den, out=np.zeros_like(num, dtype=float), where=den > 0)).round(3)

,accion,misterio,rol,terror,coseno_matricial
0,1,0,1,0,0.277
1,1,0,0,0,0.440
2,1,0,0,1,0.237
3,0,1,1,0,0.345


**Idea de arquitectura:** la similitud completa solo se calcula sobre un **prefiltro barato** (en MongoDB: `{"tags": {"$in": gustos}}` con índice), no sobre todo el catálogo.

# Parte E. Calidad y rendimiento

## 22. Validación de datos
Verifiquen sus supuestos con `assert` (el programa se detiene si algo no se cumple).

In [35]:
docs = pd.DataFrame({"_id": ["s1", "s2", "s3"], "tags": [["a"], ["b", "c"], ["a"]], "nombre": ["x", "y", "z"]})

assert docs["_id"].is_unique,            "ids repetidos"
assert docs["nombre"].notna().all(),      "hay nombres nulos"
assert docs["tags"].map(lambda t: isinstance(t, list)).all(), "tags debe ser lista"
assert docs["tags"].map(len).between(1, 8).all(), "entre 1 y 8 tags"

vocab_ok = {"a", "b", "c"}
fuera = set(e for lst in docs["tags"] for e in lst) - vocab_ok
assert not fuera, f"tags fuera de vocabulario: {fuera}"
print("OK: todas las validaciones pasaron")

# Comprobar integridad referencial (todas las referencias existen)
inter = pd.DataFrame({"objetivo_id": ["s1", "s9"]})
huerfanos = inter[~inter["objetivo_id"].isin(docs["_id"])]
print("huérfanos:\n", huerfanos)

OK: todas las validaciones pasaron
huérfanos:
   objetivo_id
1          s9


## 23. Rendimiento
Reglas de oro:
1. **Vectoriza** (operaciones sobre columnas) antes que `apply(axis=1)` o bucles.
2. Lee solo lo necesario (`usecols`, `nrows`, `chunksize`).
3. `category` para columnas con pocos valores repetidos.
4. Reduce tipos (`int32`, `float32`) cuando sea seguro.
5. No construyas un DataFrame agregando filas en un bucle: acumula en una lista y crea el DataFrame una vez.

In [36]:
import time
big = pd.DataFrame({"x": np.arange(200_000), "g": np.random.default_rng(0).choice(list("abcde"), 200_000)})

t0 = time.time(); r1 = big.apply(lambda f: f["x"] * 2, axis=1);  t1 = time.time() - t0
t0 = time.time(); r2 = big["x"] * 2;                              t2 = time.time() - t0
print(f"apply por fila: {t1:.2f}s | vectorizado: {t2:.4f}s")

m0 = big.memory_usage(deep=True).sum()
big["g"] = big["g"].astype("category"); big["x"] = pd.to_numeric(big["x"], downcast="integer")
print("memoria antes/después (MB):", round(m0 / 1e6, 1), round(big.memory_usage(deep=True).sum() / 1e6, 1))

# Acumular en lista y crear el DataFrame una sola vez
filas = [{"i": i, "v": i * i} for i in range(1000)]
pd.DataFrame(filas).tail(2)

apply por fila: 0.82s | vectorizado: 0.0017s
memoria antes/después (MB): 3.4 1.0


,i,v
998,998,996004
999,999,998001


# Parte F. MongoDB con pymongo

## 24. Conceptos mínimos
- **Documento** = diccionario JSON (BSON) · **Colección** = conjunto de documentos (≈ tabla) · **Base** = conjunto de colecciones.
- No hay esquema fijo: cada documento puede tener campos distintos.
- **Embeber** lo propio y acotado (ej: lista de tags dentro del stand); **referenciar** lo compartido o lo que crece sin tope (ej: interacciones apuntan por `_id`).
- `_id` es único; si no lo das, Mongo crea un `ObjectId`.
- Consultas = filtros tipo JSON: `{"campo": valor}`, `{"campo": {"$in": [..]}}`, `{"n": {"$gt": 5}}`.

Instalación: MongoDB local (`mongod`) o un cluster gratuito en **MongoDB Atlas**. Guarda la URI en un archivo `.env` (no la subas a git) y léela con `python-dotenv`:
```python
from dotenv import load_dotenv; import os
load_dotenv(); URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
```

In [37]:
# Conexión tolerante: si no hay servidor, el notebook sigue funcionando (las celdas Mongo se saltan)
import os
HAY_MONGO = False
try:
    from pymongo import MongoClient, ASCENDING, DESCENDING, TEXT, InsertOne, UpdateOne
    from bson import ObjectId
    URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
    cli = MongoClient(URI, serverSelectionTimeoutMS=1500)
    cli.admin.command("ping")                    # falla si no hay servidor
    db = cli["tutorial_demo"]
    HAY_MONGO = True
    print("Conectado a MongoDB")
except Exception as e:
    print("Sin MongoDB disponible (", type(e).__name__, "). Las celdas de Mongo se mostrarán como código de referencia.")

Sin MongoDB disponible ( ServerSelectionTimeoutError ). Las celdas de Mongo se mostrarán como código de referencia.


## 25. Preparar documentos desde pandas
Mongo/BSON **no acepta** `NaN`, tipos de NumPy ni `pd.Timestamp` sin convertir. Esta función los deja listos.

In [38]:
def a_documentos(d: pd.DataFrame) -> list[dict]:
    """DataFrame -> lista de dicts válidos para pymongo.
    - NaN/NaT/pd.NA -> None
    - numpy.int64/float64/bool_ -> int/float/bool de Python
    - pd.Timestamp -> datetime de Python
    - listas/dicts se respetan (recursivamente)"""
    def limpiar(v):
        if isinstance(v, (list, tuple, np.ndarray)):
            return [limpiar(x) for x in v]
        if isinstance(v, dict):
            return {str(k): limpiar(x) for k, x in v.items()}
        if v is None or v is pd.NaT or v is pd.NA:
            return None
        if isinstance(v, float) and math.isnan(v):
            return None
        if isinstance(v, pd.Timestamp):
            return v.to_pydatetime()
        if isinstance(v, np.generic):
            return v.item()
        return v
    return [{k: limpiar(v) for k, v in fila.items()} for fila in d.to_dict(orient="records")]

prueba = pd.DataFrame({
    "_id": ["s1", "s2"],
    "nombre": ["Acme", None],
    "n": np.array([1, 2], dtype="int64"),
    "x": [1.5, np.nan],
    "tags": [["a", "b"], ["c"]],
    "creado": pd.to_datetime(["2026-11-01 10:00", "2026-11-01 11:00"]),
})
docs = a_documentos(prueba)
docs

[{'_id': 's1',
  'nombre': 'Acme',
  'n': 1,
  'x': 1.5,
  'tags': ['a', 'b'],
  'creado': datetime.datetime(2026, 11, 1, 10, 0)},
 {'_id': 's2',
  'nombre': None,
  'n': 2,
  'x': None,
  'tags': ['c'],
  'creado': datetime.datetime(2026, 11, 1, 11, 0)}]

In [39]:
# Verificar que es serializable (si json.dumps pasa, BSON también, salvo datetime)
json.dumps([{k: (v.isoformat() if hasattr(v, "isoformat") else v) for k, v in d.items()} for d in docs])[:120]

'[{"_id": "s1", "nombre": "Acme", "n": 1, "x": 1.5, "tags": ["a", "b"], "creado": "2026-11-01T10:00:00"}, {"_id": "s2", "'

## 26. CRUD con pymongo
`insert_one/many` · `find/find_one` · `update_one/many` · `replace_one` · `delete_one/many` · `count_documents` · `bulk_write`.

In [40]:
if HAY_MONGO:
    col = db["stands"]
    col.drop()                                              # empezar limpio (solo en demo)

    # CREATE
    col.insert_one({"_id": "s0", "nombre": "Solo", "tags": ["accion"]})
    col.insert_many(docs, ordered=False)                    # ordered=False: sigue aunque uno falle

    # READ
    print(col.count_documents({}))
    print(col.find_one({"_id": "s1"}))
    for d in col.find({"tags": "a"}, {"_id": 1, "nombre": 1}).sort("_id", ASCENDING).limit(5):  # proyección + orden + límite
        print(d)
    print(pd.DataFrame(list(col.find({}, {"_id": 0}))))     # de Mongo a pandas

    # UPDATE
    col.update_one({"_id": "s1"}, {"$set": {"nombre": "Acme 2"}})
    col.update_many({}, {"$addToSet": {"tags": "nuevo"}})   # agrega sin repetir ($push repite)
    col.update_one({"_id": "s0"}, {"$inc": {"visitas": 1}}, upsert=True)  # contador atómico; upsert crea si no existe
    col.update_one({"_id": "s0"}, {"$pull": {"tags": "accion"}})          # quitar de un array

    # DELETE
    col.delete_one({"_id": "s2"})
    print(col.count_documents({}))

    # BULK: muchas operaciones en un viaje
    col.bulk_write([InsertOne({"_id": "b1", "tags": ["x"]}),
                    UpdateOne({"_id": "b1"}, {"$set": {"nombre": "bulk"}})])
else:
    print("Referencia (requiere MongoDB): insert_one / insert_many / find / update_one / delete_one / bulk_write")

Referencia (requiere MongoDB): insert_one / insert_many / find / update_one / delete_one / bulk_write


In [41]:
# Carga de DataFrames GRANDES por lotes (evita documentos > 16 MB por lote y reduce memoria)
def insertar_por_lotes(col, documentos, lote=5000):
    for i in range(0, len(documentos), lote):
        col.insert_many(documentos[i:i + lote], ordered=False)

# Idempotencia: si recargas, evita duplicados usando _id propio o upsert
#   from pymongo import ReplaceOne
#   col.bulk_write([ReplaceOne({"_id": d["_id"]}, d, upsert=True) for d in documentos])
# Manejo de duplicados:
#   from pymongo.errors import BulkWriteError, DuplicateKeyError
#   try: col.insert_many(docs)
#   except BulkWriteError as e: print(e.details["writeErrors"][:1])

## 27. Consultas y agregación
Operadores de consulta: `$eq $ne $gt $gte $lt $lte $in $nin $and $or $not $exists $regex $elemMatch $all $size`.
Pipeline de agregación: `$match → $group → $sort → $project → $limit → $unwind → $lookup → $addFields → $count`.

In [42]:
if HAY_MONGO:
    col = db["stands"]; col.drop()
    col.insert_many([
        {"_id": "s1", "nombre": "A", "dominio": "juego", "tags": ["accion", "rol"]},
        {"_id": "s2", "nombre": "B", "dominio": "anime", "tags": ["accion"]},
        {"_id": "s3", "nombre": "C", "dominio": "anime", "tags": ["terror", "misterio"]},
    ])
    gustos = ["accion", "misterio"]

    # Prefiltro de recomendación: comparte al menos un tag
    print([d["_id"] for d in col.find({"tags": {"$in": gustos}})])
    # Comparte TODOS los tags
    print([d["_id"] for d in col.find({"tags": {"$all": ["accion", "rol"]}})])
    # Combinaciones
    print([d["_id"] for d in col.find({"$or": [{"dominio": "juego"}, {"tags": "terror"}]})])
    # Texto parcial (regex, insensible a mayúsculas)
    print([d["_id"] for d in col.find({"nombre": {"$regex": "^a", "$options": "i"}})])

    # Agregación: ¿cuántos stands por tag?  ($unwind abre el array)
    for r in col.aggregate([{"$unwind": "$tags"},
                            {"$group": {"_id": "$tags", "frecuencia": {"$sum": 1}}},
                            {"$sort": {"frecuencia": -1}}]):
        print(r)

    # $lookup = JOIN entre colecciones
    db["interacciones"].drop()
    db["interacciones"].insert_many([{"asistente_id": "u1", "objetivo_id": "s1", "tipo": "visita"},
                                     {"asistente_id": "u1", "objetivo_id": "s3", "tipo": "favorito"}])
    pipeline = [
        {"$match": {"asistente_id": "u1", "tipo": "favorito"}},
        {"$lookup": {"from": "stands", "localField": "objetivo_id", "foreignField": "_id", "as": "stand"}},
        {"$unwind": "$stand"},
        {"$project": {"_id": 0, "stand": "$stand.nombre", "tags": "$stand.tags"}},
    ]
    print(list(db["interacciones"].aggregate(pipeline)))
else:
    print("Referencia: find({'tags': {'$in': gustos}}) · aggregate([$unwind, $group, $sort]) · $lookup para joins")

Referencia: find({'tags': {'$in': gustos}}) · aggregate([$unwind, $group, $sort]) · $lookup para joins


## 28. Índices y `explain()`
Un **índice** es una estructura ordenada que evita recorrer toda la colección (`COLLSCAN`) y permite buscar directo (`IXSCAN`).

| Consulta típica | Índice sugerido | Tipo |
|---|---|---|
| igualdad en un campo (`{"asistente_id": x}`) | `create_index("asistente_id")` | simple |
| campo array (`{"tags": {"$in": [...]}}`) | `create_index("tags")` | **multikey** (automático con arrays) |
| filtro + orden (`dominio` y `inicio`) | `create_index([("dominio",1),("inicio",1)])` | **compuesto** (orden importa: igualdad primero, rango/orden después) |
| búsqueda por palabras en texto | `create_index([("nombre","text")])` | **texto** |
| valor único | `create_index("campo", unique=True)` | **único** |
| solo documentos con el campo | `partialFilterExpression=...` | **parcial** |
| expiración automática | `expireAfterSeconds=...` | **TTL** |

**Método para justificar un índice:** medir sin índice → crear → medir con índice. Compara `totalDocsExamined`, `nReturned` y `executionTimeMillis`.

In [43]:
if HAY_MONGO:
    import time
    inter = db["inter_demo"]; inter.drop()
    rng = np.random.default_rng(1)
    inter.insert_many([{"asistente_id": f"u{rng.integers(0, 2000)}", "objetivo_id": f"s{rng.integers(0, 300)}",
                        "tipo": str(rng.choice(["visita", "asistencia", "favorito"]))} for _ in range(100_000)],
                      ordered=False)

    def medir(col, filtro):
        e = col.find(filtro).explain()["executionStats"]
        return {"examinados": e["totalDocsExamined"], "devueltos": e["nReturned"], "ms": e["executionTimeMillis"]}

    q = {"asistente_id": "u42"}
    print("SIN índice:", medir(inter, q))
    inter.create_index("asistente_id")
    print("CON índice:", medir(inter, q))

    # Índice compuesto: igualdad primero, luego orden
    inter.create_index([("objetivo_id", ASCENDING), ("tipo", ASCENDING)])
    print("compuesto:", medir(inter, {"objetivo_id": "s7", "tipo": "favorito"}))

    print([i["name"] for i in inter.list_indexes()])   # listar índices
    # inter.drop_index("asistente_id_1")                # borrar
    plan = inter.find(q).explain()["queryPlanner"]["winningPlan"]
    print(json.dumps(plan, default=str)[:200])          # busca "IXSCAN" o "COLLSCAN"
    inter.drop()
else:
    print("""Referencia:
  col.find(q).explain()['executionStats']  ->  totalDocsExamined, nReturned, executionTimeMillis
  col.create_index('campo'); col.create_index([('a',1),('b',-1)]); col.list_indexes(); col.drop_index(nombre)""")

Referencia:
  col.find(q).explain()['executionStats']  ->  totalDocsExamined, nReturned, executionTimeMillis
  col.create_index('campo'); col.create_index([('a',1),('b',-1)]); col.list_indexes(); col.drop_index(nombre)


**Reglas prácticas:** indexa lo que **filtras u ordenas con frecuencia** · cada índice acelera lecturas pero **ralentiza escrituras** y usa memoria · un índice compuesto `(a, b)` también sirve para consultas solo por `a` · confirma con `explain()` que realmente se usa.

## 29. Réplicas y sharding (conceptos para justificar la arquitectura)
- **Replica set**: varios nodos con los mismos datos (1 primario + secundarios). Si el primario cae, se elige otro automáticamente → **alta disponibilidad / tolerancia a fallos**. Las lecturas se pueden repartir con `readPreference="secondaryPreferred"`.
- **Sharding**: partir una colección en **fragmentos** repartidos entre servidores → **escalar horizontalmente** cuando una máquina no basta.
- **Clave de shard (shard key)**: campo que decide dónde va cada documento. Buena clave = **alta cardinalidad**, reparte la carga de forma pareja y aparece en las consultas más frecuentes. Mala clave: poco variada (`tipo`) o monótona (`fecha`, concentra escrituras en un shard).
  - Ejemplo razonado: en `interacciones` (crece sin tope), `asistente_id` reparte bien y la consulta "perfil de un asistente" cae en un solo shard.
  - Estrategias: **hashed** (reparto parejo) vs **ranged** (consultas por rango eficientes).
- Para colecciones pequeñas y estables (catálogo) basta con réplicas; el sharding se justifica en las que **crecen sin tope**.

```javascript
// mongosh (referencia)
sh.enableSharding("gamecon")
sh.shardCollection("gamecon.interacciones", { asistente_id: "hashed" })
rs.status()   // estado del replica set
```
```python
# pymongo con réplicas y escritura segura
MongoClient("mongodb://h1,h2,h3/?replicaSet=rs0", w="majority", readPreference="secondaryPreferred")
```

# Parte G. Cierre

## 30. Plantilla de pipeline ETL reutilizable
Estructura recomendada: una función por etapa, entradas/salidas claras y validación al final.

In [44]:
def extraer(ruta: Path) -> pd.DataFrame:
    """Lee la fuente cruda. Nunca modifica el archivo original."""
    return pd.read_csv(ruta)

def transformar(df: pd.DataFrame, mapeo: dict[str, str]) -> pd.DataFrame:
    """Limpia, normaliza y construye la unidad final (una fila por entidad)."""
    out = (df.dropna(subset=["empresa", "generos"])
             .assign(empresa=lambda d: normalizar(d["empresa"]),
                     generos=lambda d: d["generos"].str.split(","))
             .explode("generos"))
    out["generos"] = normalizar(out["generos"])
    out["tag"] = out["generos"].map(mapeo)
    sin_mapa = out.loc[out["tag"].isna(), "generos"].unique()
    if len(sin_mapa):
        print("Sin mapear:", sin_mapa)
    return (out.dropna(subset=["tag"])
               .groupby("empresa")
               .agg(tags=("tag", lambda s: sorted(set(s))))
               .reset_index())

def validar(df: pd.DataFrame) -> None:
    assert df["empresa"].is_unique
    assert df["tags"].map(len).ge(1).all()

def cargar(df: pd.DataFrame, col) -> None:
    documentos = a_documentos(df)
    col.insert_many(documentos, ordered=False)

# Uso:  df = extraer(RAW/'x.csv'); res = transformar(df, dic); validar(res); cargar(res, db['stands'])
demo = transformar(toy.drop_duplicates(), {"action": "accion", "fantasy": "rol", "sci-fi": "ciencia-ficcion", "drama": "drama"})
validar(demo); demo

,empresa,tags
0,acme,"[accion, rol]"
1,omega,[drama]
2,zeta-co,"[accion, ciencia-ficcion, drama]"


## 31. Ejercicios (con soluciones al final de cada bloque para auto-corregir)
Cada ejercicio es genérico; resuélvanlos con sus propios datos cuando puedan.

1. **Perfilado.** Dado un DataFrame, escribe una función que devuelva las 5 columnas con más nulos y las columnas con un único valor.
2. **Normalización.** Convierte `["  Sci-Fi ", "ROLE_PLAYING", "Acción/Aventura"]` en `["sci-fi", "role-playing", "accion-aventura"]`.
3. **Explode.** Dada una columna `"a, b; c"`, obtén una fila por elemento separando por `,` y `;` (pista: `str.split(r"[,;]")`).
4. **Mapeo y auditoría.** Mapea una lista de géneros con un diccionario y muestra cuáles no se pudieron mapear con su frecuencia.
5. **Agrupar.** Crea una tabla de una fila por empresa con: nº de ítems, y lista ordenada sin repetidos de etiquetas.
6. **Merge seguro.** Une dos tablas con `validate="many_to_one"` y comprueba que no cambió el nº de filas.
7. **Similitud.** Implementa Jaccard ponderado por IDF y rankea 5 ítems para un usuario con 3 gustos.
8. **Sintéticos.** Genera 1000 interacciones donde el 10% de los stands reciba el 70% de las visitas.
9. **Mongo.** Carga el resultado del 5, crea un índice sobre `tags` y compara `totalDocsExamined` con y sin índice.
10. **Validación.** Escribe asserts que verifiquen ids únicos, tags dentro de un vocabulario y referencias existentes.

In [45]:
# Soluciones breves (tapar antes de intentar)
# 1
def nulos_y_constantes(d):
    return d.isna().sum().nlargest(5), d.columns[d.nunique(dropna=True) <= 1].tolist()
print(nulos_y_constantes(toy))
# 2
print(normalizar(pd.Series(["  Sci-Fi ", "ROLE_PLAYING", "Acción/Aventura"])).tolist())
# 3
pd.Series(["a, b; c"]).str.split(r"[,;]\s*", regex=True).explode().tolist()
# 6
izq = pd.DataFrame({"k": [1, 2, 2]}); der = pd.DataFrame({"k": [1, 2], "y": ["u", "v"]})
x = izq.merge(der, on="k", how="left", validate="many_to_one"); assert len(x) == len(izq)
# 8
rng = np.random.default_rng(0); n_st = 100; top = int(0.1 * n_st)
p = np.r_[np.full(top, 0.7 / top), np.full(n_st - top, 0.3 / (n_st - top))]
vis = rng.choice(n_st, size=1000, p=p); print((vis < top).mean())

(nombre     1
empresa    1
anio       1
generos    1
ventas     0
dtype: int64, [])
['sci-fi', 'role-playing', 'accion-aventura']
0.678


## 32. Glosario de errores frecuentes
| Síntoma | Causa probable | Solución |
|---|---|---|
| `KeyError: 'col'` | nombre mal escrito o con espacios | `df.columns.tolist()`, `df.columns = df.columns.str.strip()` |
| `ValueError: truth value ambiguous` | usar `and`/`or`/`if serie` | usa `&`, `|`, `.any()`, `.all()` |
| Filtro devuelve vacío | condición sobre texto sin normalizar | normaliza antes (`strip`, `lower`) |
| `TypeError: unhashable type: 'list'` | `drop_duplicates`/`groupby` sobre columna de listas | convierte a tupla: `.map(tuple)` |
| Filas se multiplican tras `merge` | claves duplicadas | `validate=`, `drop_duplicates` en la clave |
| `AttributeError: Can only use .str accessor` | columna no es texto | `.astype("string")` |
| `bson.errors.InvalidDocument: cannot encode numpy.int64` | tipos NumPy | `a_documentos()` (sección 25) |
| `DocumentTooLarge` | documento > 16 MB | referenciar en lugar de embeber |
| `DuplicateKeyError` | `_id` repetido al recargar | `drop()` previo o `ReplaceOne(..., upsert=True)` |
| `ServerSelectionTimeoutError` | Mongo no está corriendo / URI mala | arrancar `mongod`, revisar URI/IP permitida en Atlas |
| `explain` muestra `COLLSCAN` | falta índice o no coincide con la consulta | `create_index` con los campos del filtro |
| Resultados distintos en cada ejecución | aleatorios sin semilla | `np.random.default_rng(42)` |
| Kernel muere cargando archivo | archivo demasiado grande | `usecols`, `nrows`, `chunksize`, `dtype` |

**Fin.** Con las partes A–F pueden construir un pipeline completo: leer → limpiar → normalizar → unir/agrupar → generar sintéticos → medir similitud → cargar a MongoDB con índices justificados.